# 07 · Left sources on a real pair

This notebook drives `dtrack_left.backends.sources` against the Left database of one of your real pairs: `source_from_config`, the Oracle/Athena/File/DuckDB source classes, `preflight`, `query`, `partition_counts`, `pull_window` and `write_rows`.

It mirrors `tests/unit/dtrack_left/backends/test_sources.py`, but instead of a fake Oracle driver and synthetic CSVs it talks to the real connection named by your pairs config. It only runs `SELECT` statements; the only files it writes go to a local WORKDIR.

## How to approach

**Prerequisites**
- Kernel: the repo `.venv` (`uv sync`), plus the driver extra for your Left side: Oracle needs `oracledb` (base install), LDAP lookups need `uv sync --extra ldap`, Athena needs `uv sync --extra athena`. File/DuckDB sources only need `duckdb`.
- The project `.env` must hold the Left credentials by macro name: `<macro>_USR`, `<macro>_PWD`, `<macro>_DSN` (or `DTRACK_ORACLE_MACROS` + `LDAP_DSN` for service lookups). Athena uses your normal AWS credentials (`AWS_PROFILE`).
- `DTRACK_PROD_CONFIG` points at your real pairs JSON; `DTRACK_NB_PAIR` picks the pair.

**Run order**: top to bottom. Sections 1-2 never connect. Sections 3-6 run cheap queries (`WHERE 1=0`, a grouped count over `DAYS` days). Sections 7-8 pull real rows; they skip themselves when a day is bigger than `MAX_ROWS`.

**What to look at**: `preflight()` must say `ok: True`; every `col_map` left name should exist in the column list; the one-day pull must have exactly as many rows as `partition_counts` reported for that day.

**Failure modes**
- `Oracle requires user, password, and dsn` - the `<macro>_*` variables are not in the `.env` that was loaded (check the "env file" line of the setup cell).
- `ORA-00942` in preflight - wrong table name or no grant for this user.
- Empty `partition_counts` - wrong `date_col` / `date_type`, or no data in the window (move `TO_DATE`).
- Pull rows != counted rows - the `date_type` makes `partition_counts` and `pull_window` filter differently (for example a timestamp column declared as a string).

In [ ]:
import os
from pathlib import Path

CONFIG_PATH = os.environ.get("DTRACK_PROD_CONFIG", "C:/work/dtrack/c.json")   # your real pairs config
PAIR = os.environ.get("DTRACK_NB_PAIR", "orders")                             # pair to drive
ENV_FILE = os.environ.get("DTRACK_ENV_FILE", "")                              # "" = search for .env upward from the cwd
TO_DATE = None                                                                # None = yesterday; or "YYYY-MM-DD"
DAYS = 3                                                                      # window = DAYS days ending at TO_DATE
MAX_ROWS = int(os.environ.get("DTRACK_PROD_MAX_ROWS", "200000"))              # skip pulls above this many rows
PREFLIGHT_ALL = False                                                         # True = also preflight every active pair
WORKDIR = Path.home() / ".local/plans/nbs/_work/07"                           # local output folder
KEEP = False                                                                  # True = keep WORKDIR after the run

In [ ]:
import json
import re
import shutil
from datetime import date, timedelta

import pandas as pd
from dotenv import find_dotenv, load_dotenv

from dtrack_left.cli.config import load_config

# load the project .env first so <macro>_USR / _PWD / _DSN are visible to the sources
env_file = ENV_FILE or find_dotenv(usecwd=True)
print("env file:", env_file or "(none found)")
if env_file:
    load_dotenv(env_file, override=False)

# the real pairs config: conn_macro profiles expanded, col_map files read, validated
config = load_config(CONFIG_PATH)
active = [name for name, spec in config["pairs"].items() if not spec.get("skip")]
print(f"{len(active)} active pair(s):", active)

# a private copy of the pair with the check window attached
pair = json.loads(json.dumps(config["pairs"][PAIR]))
TO = TO_DATE or (date.today() - timedelta(days=1)).isoformat()
FROM = (date.fromisoformat(TO) - timedelta(days=DAYS - 1)).isoformat()
pair["fromDate"], pair["toDate"] = FROM, TO
print(f"pair {PAIR}: left source={pair['left'].get('source')} window {FROM}..{TO}")

WORKDIR.mkdir(parents=True, exist_ok=True)


# >>> show only the edges of a secret-ish value <<< #
def mask(value):
    if not value:
        return value
    text = str(value)
    return text[:2] + "***" + text[-2:] if len(text) > 4 else "***"

## 1. Pick the source class (`source_from_config`)

`source_from_config` dispatches on `left.source`: `oracle`, `athena`/`aws`, `file`/`csv`, `duckdb`; `hadoop` is deliberately disabled and unknown names raise. Building a source never connects, so this loop is safe over every pair.

In [ ]:
from dtrack_left.backends import sources
from dtrack_left.backends.sources import (
    AthenaSource,
    DuckDBSource,
    FileSource,
    OracleSource,
    source_from_config,
    write_rows,
)

# what each active pair's Left side turns into; disabled/unknown sources show their error
for name in active:
    spec = config["pairs"][name]["left"]
    try:
        built = source_from_config(spec)
        print(f"{name:32} {str(spec.get('source')):8} -> {type(built).__name__:13} dialect={built.dialect}")
    except Exception as exc:
        print(f"{name:32} {str(spec.get('source')):8} -> {type(exc).__name__}: {exc}")

# the source this notebook drives, and the relation every query selects FROM
source = source_from_config(pair["left"])
print()
print("class   :", type(source).__name__, "| dialect:", source.dialect)
print("relation:", source.relation())
print("where   :", pair["left"].get("where") or "(none)")

## 2. Connection settings each source resolved

Oracle credentials resolve in this order: explicit field, `*_env` variable name, `<conn_macro>_USR/_PWD/_DSN`, then `ORACLE_*`. A `service` (or `DTRACK_ORACLE_MACROS`) plus an `LDAP_DSN` template triggers an LDAP lookup in `resolve_dsn()`. Athena and file sources show their own settings. Secrets are masked.

In [ ]:
# Oracle: which user/dsn/service came out of the .env, and what DSN the driver will get
if isinstance(source, OracleSource):
    print("conn_macro          :", source.macro or "(none)")
    print("user                :", mask(source.user))
    print("password set        :", bool(source.password))
    print("dsn                 :", source.dsn)
    print("service             :", source.service)
    print("DTRACK_ORACLE_MACROS:", sources.oracle_macro_services())
    print("LDAP_DSN template   :", os.getenv("LDAP_DSN") or "(unset)")
    # this may contact the LDAP server when service + LDAP_DSN are set
    print("resolve_dsn()       :", source.resolve_dsn())

# Athena: the staging dir and region pyathena will use (credentials come from boto3)
elif isinstance(source, AthenaSource):
    print("database      :", source.database)
    print("s3_staging_dir:", source.s3_staging_dir)
    print("region        :", source.region)
    print("AWS_PROFILE   :", os.getenv("AWS_PROFILE") or "(default chain)")

# File: the DuckDB reader call built from path/format/options
elif isinstance(source, FileSource):
    print("path   :", source.path)
    print("format :", source.format)
    print("reader :", source.reader())
    print("date_type forced to 'date' by date_format:", source.config.get("date_type"))

# DuckDB database file: attached read-only under its file stem
elif isinstance(source, DuckDBSource):
    print("path    :", source.path)
    print("alias   :", source.alias)
    print("snapshot:", source.snapshot)

## 3. Preflight

`preflight()` runs `SELECT * FROM <relation> WHERE 1=0` and turns any error into `{"ok": False, "error": ...}` instead of raising. File sources also check that `date_col` exists.

In [ ]:
# the empty select proves login, grants and the relation name, without reading rows
print("SQL:", f"SELECT * FROM {source.relation()} WHERE 1=0")
health = source.preflight()
print(health)

# optional sweep: the same check over every active pair
if PREFLIGHT_ALL:
    for name in active:
        try:
            result = source_from_config(config["pairs"][name]["left"]).preflight()
        except Exception as exc:
            result = {"ok": False, "error": f"{type(exc).__name__}: {exc}"}
        print(f"{name:32} {result}")

## 4. Columns via `query()`

`query(sql)` opens a connection, runs one statement and returns `(columns, rows)`, closing the cursor and connection. Here it lists the real columns so you can see whether every `col_map` left name, the `date_col` and the key columns exist.

In [ ]:
# real column names from the database
columns, _ = source.query(f"SELECT * FROM {source.relation()} WHERE 1=0")
print(len(columns), "columns:", columns)

# every name the pair relies on, matched exactly and case-insensitively
date_col = pair["left"]["date_col"]
keys = [part.strip() for part in str(pair.get("key") or "").split(",") if part.strip()]
wanted = [date_col, *keys, *pair["col_map"]]
lower = {name.lower(): name for name in columns}
report = pd.DataFrame(
    [
        {"name": name, "exact": name in columns, "db_spelling": lower.get(name.lower(), "MISSING")}
        for name in dict.fromkeys(wanted)
    ]
)
report

## 5. Partition counts over the window

`partition_counts(from, to)` groups by the partition key (`TRUNC(col)` / `CAST(col AS DATE)` when `date_type` is date/timestamp) and canonicalises every key to `YYYY-MM-DD`. Below is the exact SQL it builds, then the counts.

In [ ]:
from dtrack_left.common.dates import bounds_where, canonical_date, contiguous_runs, day_expr, partition_where

# rebuild the SQL partition_counts runs, from the same helpers
date_type = pair["left"].get("date_type")
date_ref = day_expr(date_col, source.dialect, date_type)
bounds = bounds_where(date_col, dialect=source.dialect, date_type=date_type, from_date=FROM, to_date=TO)
count_sql = (
    f"SELECT {date_ref} AS dtrack_partition, COUNT(*) AS dtrack_count "
    f"FROM {source.relation()} WHERE {source._where(bounds)} GROUP BY {date_ref}"
)
print(count_sql)

# the real counts, day by day
counts = source.partition_counts(FROM, TO)
print(f"{len(counts)} day(s), {sum(counts.values())} row(s)")
pd.Series(counts, name="rows").sort_index().to_frame()

In [ ]:
# an open-ended window (from only) must agree with the closed one on the overlapping days
open_ended = source.partition_counts(TO)
print("from", TO, "onward:", dict(sorted(open_ended.items())))
print("same rows for", TO, ":", open_ended.get(TO) == counts.get(TO))

## 6. Raw partition values and `canonical_date`

The database may return dates, timestamps, or strings like `20260102`. `partition_counts` folds them with `canonical_date`, adding up counts that land on the same day. This shows the raw values next to their canonical day.

In [ ]:
# the same grouped query, but keep the raw partition values
_, raw_rows = source.query(count_sql)
raw = pd.DataFrame(
    [
        {"raw": value, "python_type": type(value).__name__, "canonical": canonical_date(value), "rows": int(n)}
        for value, n in raw_rows
    ]
).sort_values("canonical")
raw

In [ ]:
# for reference only: the text formats canonical_date accepts (literal examples, not your data)
pd.DataFrame(
    [{"input": text, "canonical": canonical_date(text)} for text in ("2026-01-02", "20260102", "01/02/2026", "02JAN2026", "02-Jan-2026")]
)

## 7. Pull the smallest day (`pull_window`)

`pull_window(partitions, columns, output, key_columns=...)` selects `date_col`, then the keys, then the columns (each once), filters to the given days and writes a CSV through `write_rows`. The row count must equal the day's count from section 5.

In [ ]:
# smallest day in the window keeps the pull cheap
if not counts:
    print("no Left rows in the window; move TO_DATE")
else:
    day, expected = min(counts.items(), key=lambda item: item[1])
    pull_columns = [name for name in pair["col_map"] if name.lower() != date_col.lower()]
    print(f"day {day}: {expected} row(s); {len(pull_columns)} column(s); keys {keys}")
    print("filter:", partition_where(date_col, [day], dialect=source.dialect, date_type=date_type))
    if expected > MAX_ROWS:
        print(f"skipped: {expected} > MAX_ROWS={MAX_ROWS}")
    else:
        one_day = source.pull_window([day], pull_columns, WORKDIR / "left_one_day.csv", key_columns=keys)
        frame = pd.read_csv(one_day, dtype=object, keep_default_na=False)
        print("file   :", one_day)
        print("header :", list(frame.columns)[:8], "...")
        print("rows   :", len(frame), "| expected:", expected, "| equal:", len(frame) == expected)
        display(frame.head())

## 8. Pull several days with a gap

A non-contiguous set of days becomes one `OR` clause per contiguous run (`contiguous_runs`). Only the date and key columns are pulled, then rows per day are compared with the counts.

In [ ]:
# every other day of the window, so the runs have gaps when DAYS >= 3
picked = sorted(counts)[::2]
total = sum(counts[day] for day in picked)
print("days  :", picked)
print("runs  :", contiguous_runs(picked))
print("filter:", partition_where(date_col, picked, dialect=source.dialect, date_type=date_type))
if not picked or total > MAX_ROWS:
    print(f"skipped: {total} row(s), MAX_ROWS={MAX_ROWS}")
else:
    gapped = source.pull_window(picked, [], WORKDIR / "left_gapped.csv", key_columns=keys)
    frame_gap = pd.read_csv(gapped, dtype=object, keep_default_na=False)
    per_day = frame_gap[date_col].map(canonical_date).value_counts().to_dict()
    display(pd.DataFrame([{"day": d, "pulled": per_day.get(d, 0), "counted": counts[d]} for d in picked]))

## 9. Raw text survives, and `write_rows`

Values travel as the driver returns them; for file sources everything is read as text, so `007` or `1.50` stay as written. `write_rows(path, columns, rows)` is the CSV writer behind `pull_window`; here it re-writes a few real rows.

In [ ]:
# look for values whose text would change if parsed as numbers (leading zeros, trailing zeros)
if "frame" in globals():
    pattern = re.compile(r"^0\d|\.\d*0$")
    for name in frame.columns:
        hits = frame[name][frame[name].str.contains(pattern)].head(3).tolist()
        if hits:
            print(f"{name:30} {hits}")

    # write the first five pulled rows again with write_rows and show the bytes
    sample = write_rows(WORKDIR / "sample" / "five_rows.csv", list(frame.columns), frame.head(5).itertuples(index=False))
    print(sample)
    print(sample.read_text(encoding="utf-8"))

## Cleanup

Removes WORKDIR (the pulled CSVs) unless `KEEP = True`. Nothing was written anywhere else.

In [ ]:
# local files only
if KEEP:
    print("kept", WORKDIR)
else:
    shutil.rmtree(WORKDIR, ignore_errors=True)
    print("removed", WORKDIR)

## What to check

- Section 1: the pair's Left side builds the expected class; no other pair shows an error you did not expect.
- Section 2: user/DSN came from the right macro; `resolve_dsn()` returns a real descriptor when LDAP is used.
- Section 3: `preflight()` is `ok: True`.
- Section 4: every row says `exact = True`; a `False` with a `db_spelling` means fix the case in `col_map`/`key`.
- Sections 5-6: counts exist for the window and raw values canonicalise to the days you expect.
- Sections 7-8: pulled rows equal counted rows for every day.
- Section 9: leading/trailing zeros are preserved as text.